# Host the answer model for the Streamlit UI

Use a **GPU runtime with at least 40 GB** (A100 40 GB). Put `repo_rag_ask_source.zip` (the project's
`notebooks/repo_rag_ask_source.zip` as built, same name) at the top of My Drive, then run the cells in
order. The last output gives a **URL and a token**: paste them into the UI on your laptop
(`uv run streamlit run src/repo_rag/ui/app.py`).

Only text generation runs here. The index, retrieval, tools and citation check run on your
laptop, so no index needs uploading. Prompts and retrieved code travel through a public
Cloudflare quick tunnel to this machine; every request needs the token, but do not use this for
a private repository. The tunnel program (cloudflared 2026.9.1) is downloaded from
GitHub and checked against a pinned SHA-256 before it runs. The model is **provisional**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!command -v uv >/dev/null || pip install -q uv
# Pinned to the versions in the project's uv.lock. Colab already ships a CUDA build of torch.
!uv pip install --system -q "chromadb==1.5.9" "fastapi==0.162.0" "gitpython==3.2.1" "langchain-chroma==1.1.0" "langchain-core==1.6.13" "langchain-text-splitters==1.1.3" "langgraph==1.2.19" "pydantic==2.14.1" "pydantic-settings==2.16.1" "rank-bm25==0.2.2" "rich==15.0.0" "sentence-transformers==5.7.0" "transformers==4.57.6" "tree-sitter==0.26.0" "tree-sitter-language-pack==1.22.1" "typer==0.27.3" "uvicorn==0.54.0"

In [ ]:
import hashlib, io, os, pathlib, subprocess, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')
SRC = pathlib.Path('/content/repo_rag_src')
SOURCE = pathlib.Path('/content/drive/MyDrive/repo_rag_ask_source.zip')
EXPECTED_SHA256 = 'b63296e810f443a90f3944435d102adc2c428eb1f11b02f2678505431b7fcb71'
assert SOURCE.exists(), 'Upload repo_rag_ask_source.zip (from the project notebooks folder) to My Drive'
packed = SOURCE.read_bytes()
assert hashlib.sha256(packed).hexdigest() == EXPECTED_SHA256, 'Source ZIP differs from notebook'
with zipfile.ZipFile(io.BytesIO(packed)) as archive:
    archive.extractall(SRC)
os.environ.update(
    REPO_RAG_DATA_DIR='/content/data', REPO_RAG_COMPUTE_DEVICE='cuda',
    REPO_RAG_EMBEDDING_DTYPE='float32', TOKENIZERS_PARALLELISM='false',
)
sys.path.insert(0, str(SRC))
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
except Exception:
    pass  # optional: a Hugging Face token only makes downloads faster
print('unpacked', len(list(SRC.rglob('*.py'))), 'files')

Load the model once (a few minutes).

In [ ]:
from repo_rag.agent.local_llm import LocalChatModel
from repo_rag.config import LOCAL_14B_MODEL, LOCAL_14B_REVISION, Settings

# The default answer model is the 27B; this notebook loads the 14B locally.
settings = Settings().model_copy(
    update={'llm_model': LOCAL_14B_MODEL, 'llm_model_revision': LOCAL_14B_REVISION}
)
model = LocalChatModel(settings)
print(f'loaded {settings.llm_model} in {model.load_seconds:.0f}s on {model.hardware}')

# A second model on the same GPU re-scores commit search results (bug -> past fix). The 8B
# reranker would not fit next to the 14B answer model on 40 GB (parameter-count estimate, not
# measured), so the 4B one is used. If it does not load, the server simply runs without it.
RERANKER_MODEL = 'Qwen/Qwen3-Reranker-4B'   # set to None to serve without a reranker
reranker = None
if RERANKER_MODEL:
    import torch
    from repo_rag.retrieval.rerank import QwenReranker
    try:
        reranker = QwenReranker(settings.model_copy(update={'reranker_model': RERANKER_MODEL}))
        reranker.score('warm-up', ['warm-up'])   # loads the weights now, so a failure shows here
        print('reranker loaded:', RERANKER_MODEL)
    except Exception as error:
        reranker = None
        print('RERANKER NOT LOADED, serving without it:', type(error).__name__, str(error)[:200])
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print(f'GPU memory in use: {(total - free) / 2**30:.1f} of {total / 2**30:.1f} GiB')

Start the model server on this machine.

In [ ]:
import secrets, threading
from repo_rag.agent.remote_llm import serve_model

PORT = 8000
TOKEN = secrets.token_urlsafe(24)   # new every session; only someone holding it can generate
info = {'model': settings.llm_model, 'hardware': model.hardware,
        'max_input_tokens': str(settings.agent_max_input_tokens),
        'decode_mode': globals().get('DECODE_MODE', 'transformers')}
if reranker is not None:
    info['rerank'] = RERANKER_MODEL
server = serve_model(model, TOKEN, info, port=PORT, reranker=reranker)
threading.Thread(target=server.serve_forever, daemon=True).start()
print(f'model server listening on 127.0.0.1:{PORT}')

Open a tunnel so the laptop can reach it, and print the URL and token.

In [ ]:
import re, time, urllib.request
CLOUDFLARED_URL = ('https://github.com/cloudflare/cloudflared/releases/download/'
                   '2026.9.1/cloudflared-linux-amd64')
CLOUDFLARED_SHA256 = '03f1f25d1cc93b9ad6c60569d44060bc4f17ed97075760ed8cfca4b12dcd68cc'
binary = pathlib.Path('/content/cloudflared')
if not binary.exists():
    data = urllib.request.urlopen(CLOUDFLARED_URL).read()
    assert hashlib.sha256(data).hexdigest() == CLOUDFLARED_SHA256, 'cloudflared differs from pin'
    binary.write_bytes(data)
    binary.chmod(0o755)
tunnel = subprocess.Popen(
    [str(binary), 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
URL, deadline, log = None, time.time() + 90, []
for line in tunnel.stdout:
    log.append(line)
    found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if found:
        URL = found.group(0)
        break
    if time.time() > deadline:
        break
assert URL, 'No tunnel URL after 90 s. Last log lines:\n' + ''.join(log[-15:])
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()  # drain

from repo_rag.agent.remote_llm import RemoteChatModel
for attempt in range(30):   # a new tunnel name can take a few seconds to resolve
    try:
        print('reachable through the tunnel:', RemoteChatModel(URL, TOKEN, timeout=10).health())
        break
    except RuntimeError as error:
        last = error
        time.sleep(2)
else:
    raise last
print()
print('Paste these into the Streamlit sidebar (or .env):')
print('  URL  ', URL)
print('  Token', TOKEN)

Keep this running while you chat. It releases the GPU by itself after 10 minutes with no request (or 4 hours in total); change `IDLE_MINUTES` and `MAX_HOURS` in the cell to alter that.

In [ ]:
# Leave this cell running while you use the UI. When no question or rerank has arrived for
# IDLE_MINUTES (a request still running counts as activity), or MAX_HOURS have passed since
# this cell started, it shuts the server and tunnel down and releases the GPU by itself.
# Stop the cell (square button) to take over manually; the server then stays up until the next cell.
import time
IDLE_MINUTES = 10
MAX_HOURS = 4

started = time.monotonic()
reason = None
try:
    while reason is None:
        time.sleep(30)
        if server.idle_seconds() >= IDLE_MINUTES * 60:
            reason = f'no request for {IDLE_MINUTES} minutes'
        elif time.monotonic() - started >= MAX_HOURS * 3600:
            reason = f'{MAX_HOURS} hours reached'
except KeyboardInterrupt:
    print('stopped waiting; the server is still up until you run the next cell')
if reason:
    print('Releasing the GPU:', reason)
    from google.colab import runtime
    server.shutdown()
    tunnel.terminate()
    model.close()
    runtime.unassign()

In [ ]:
# Shut everything down and release the GPU (saves compute units).
from google.colab import runtime
server.shutdown()
tunnel.terminate()
model.close()
runtime.unassign()